# DATE 27 lớp — So sánh neural, KG-guided và fusion với baseline

Notebook Colab này tạo gói kết quả cho báo cáo ngày mai: đánh giá **một lần** checkpoint được chọn theo validation `best.pt`, lấy đủ ba nhánh `neural`, `kg` và `final`, rồi so sánh với các baseline đã công bố.

Hãy bật GPU nếu có thể. Code được clone theo URL/commit cấu hình bên dưới; dữ liệu và checkpoint có thể nằm trên Google Drive. Các số trong bài báo khác giao thức/tập test hiện tại thì chỉ dùng để tham chiếu, không coi là phép tái lập trực tiếp.

**Quy tắc checkpoint:** báo cáo này bắt buộc dùng `best.pt` (được chọn bởi validation). Không thay bằng `best_test.pt`, vì đó là lựa chọn theo test và gây rò rỉ test vào báo cáo.

## 0. Kết nối Drive và cấu hình

Chỉ cần sửa các biến trong cell kế tiếp. `DRIVE_FINAL_DIR` phải trỏ tới thư mục `final` trên Drive; notebook sẽ dùng `final/data/Input` cho split, tensor keypoint, graph và template. Kết quả mặc định ghi vào `DRIVE_FINAL_DIR.parent/results/runs/date27_baseline_comparison`; có thể đặt `RESULTS_DIR_OVERRIDE` nếu cần vị trí khác. Có thể đặt `CHECKPOINT_OVERRIDE` nếu checkpoint nằm ở vị trí khác. Đường dẫn đều là `pathlib.Path`, không ghép lệnh shell.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import datetime as dt
import hashlib
import json
import os
import subprocess
import sys
from pathlib import Path

# ===== CHỈNH CÁC GIÁ TRỊ NÀY KHI CẦN =====
DRIVE_FINAL_DIR = Path('/content/drive/MyDrive/DATE27_NeuroSymbolic/final')
DRIVE_INPUT_DIR = DRIVE_FINAL_DIR / 'data' / 'Input'
DRIVE_CHECKPOINT_DIR = DRIVE_FINAL_DIR / 'data' / 'Output' / 'checkpoints'
CHECKPOINT_SUBDIR = 'cross_attention_v4_template170_retrain_next'
CHECKPOINT_OVERRIDE = ''  # Ví dụ: /content/drive/MyDrive/.../best.pt
INPUT_DIR_OVERRIDE = ''  # Để trống để dùng DRIVE_FINAL_DIR/data/Input
TEST_FILE_NAME = 'test_1229.txt'  # Đổi thành test.txt nếu bộ dữ liệu dùng tên đó
RESULTS_DIR_OVERRIDE = ''  # Để trống để ghi vào results/runs trên Drive
LEGACY_NEURAL_METRICS_OVERRIDE = ''  # Tùy chọn; chỉ thêm benchmark nếu file metrics tồn tại
EXPECTED_VALIDATION_CHECKPOINT_SHA256 = '7faa3d4536b197851432566e366c9ab14a9b99dd5255c2a0dffbddf74081d9a5'
ALLOW_NON_CANONICAL_CHECKPOINT = False  # Chỉ bật khi báo cáo được gắn nhãn NON-CANONICAL

REPO_URL = os.environ.get(
    'DATE27_REPO_URL',
    'https://github.com/HuyenHoang28/DynamicHandGestureWithXAI.git',
)
REPO_BRANCH = os.environ.get('DATE27_REPO_BRANCH', 'HuyenHoang')
REPO_COMMIT = os.environ.get('DATE27_REPO_COMMIT', 'e817f16e0d504e95064b889288f06231bcf692ab')
WORK_ROOT = Path('/content/date27_neurosymbolic')
REPO_DIR = WORK_ROOT / 'dynamic_hand_gesture_recognition'

def choose_path(override: str, default: Path) -> Path:
    return Path(override).expanduser() if str(override).strip() else default

INPUT_DIR = choose_path(INPUT_DIR_OVERRIDE, DRIVE_INPUT_DIR)
CHECKPOINT_PATH = choose_path(
    CHECKPOINT_OVERRIDE,
    DRIVE_CHECKPOINT_DIR / CHECKPOINT_SUBDIR / 'best.pt',
)
RESULTS_DIR = choose_path(
    RESULTS_DIR_OVERRIDE,
    DRIVE_FINAL_DIR.parent / 'results' / 'runs' / 'date27_baseline_comparison',
)
LEGACY_NEURAL_METRICS_PATH = choose_path(
    LEGACY_NEURAL_METRICS_OVERRIDE,
    DRIVE_FINAL_DIR.parent / 'results' / 'runs' / 'date27_neural_only_legacy_2026-10-23' / 'test_metrics.json',
)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

EXPECTED_NUM_CLASSES = 27
BATCH_SIZE = 16
NUM_WORKERS = 0  # Ổn định khi đọc trực tiếp từ Google Drive

print('Drive final:', DRIVE_FINAL_DIR)
print('Input:', INPUT_DIR)
print('Checkpoint dự kiến:', CHECKPOINT_PATH)
print('Thư mục kết quả:', RESULTS_DIR)
print('Metrics neural-only legacy (tùy chọn):', LEGACY_NEURAL_METRICS_PATH)

## 1. Clone code theo repository và commit cấu hình

Notebook mặc định checkout commit `e817f16e0d504e95064b889288f06231bcf692ab` và vẫn ghi SHA thực tế vào manifest. Có thể đặt `DATE27_REPO_COMMIT` để chạy một commit khác, nhưng phải ghi rõ thay đổi provenance trong báo cáo.

In [ ]:
if not str(REPO_URL).strip():
    raise ValueError('REPO_URL đang trống; hãy điền URL Git trước khi clone.')

WORK_ROOT.mkdir(parents=True, exist_ok=True)
if REPO_DIR.exists():
    if not (REPO_DIR / '.git').is_dir():
        raise RuntimeError(f'{REPO_DIR} đã tồn tại nhưng không phải Git repository.')
    print('Dùng repository đã có trong runtime:', REPO_DIR)
else:
    clone_command = ['git', 'clone', '--filter=blob:none']
    if not str(REPO_COMMIT).strip() and str(REPO_BRANCH).strip():
        clone_command.extend(['--branch', REPO_BRANCH, '--single-branch'])
    clone_command.extend([REPO_URL, str(REPO_DIR)])
    subprocess.run(clone_command, check=True)

if str(REPO_COMMIT).strip():
    subprocess.run(
        ['git', '-C', str(REPO_DIR), 'fetch', '--depth', '1', 'origin', REPO_COMMIT],
        check=True,
    )
    subprocess.run(
        ['git', '-C', str(REPO_DIR), 'checkout', '--detach', REPO_COMMIT],
        check=True,
    )

REPO_COMMIT_RESOLVED = subprocess.check_output(
    ['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'],
    text=True,
).strip()
print('Repository:', REPO_URL)
print('Commit thực tế:', REPO_COMMIT_RESOLVED)

## 2. Cài dependency

Cài từ `requirements.txt` của chính commit đã clone. Lệnh dùng danh sách đối số của `subprocess`, nên đường dẫn có khoảng trắng vẫn an toàn.

In [ ]:
REQUIREMENTS_PATH = REPO_DIR / 'requirements.txt'
if not REQUIREMENTS_PATH.is_file():
    raise FileNotFoundError(f'Không tìm thấy requirements.txt: {REQUIREMENTS_PATH}')

subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '--quiet', '-r', str(REQUIREMENTS_PATH)],
    check=True,
)
print('Đã cài dependency từ:', REQUIREMENTS_PATH)

## 3. Kiểm tra dữ liệu, script và checkpoint

Cấu trúc tối thiểu trên Drive là `final/data/Input/{keypoint_tensor_cache,instance_graphs,template_graphs,*.txt}` và một checkpoint `.../best.pt`. Cell này dừng sớm nếu thiếu đường dẫn.

**Nhắc lại:** `best.pt` là checkpoint được chọn theo validation. Không chọn `best_test.pt` hoặc tên tương đương theo test. Cell kiểm tra SHA-256 của checkpoint và `test_1229.txt`; hash checkpoint phải khớp giá trị canonical đã khóa, trừ khi bật cờ `ALLOW_NON_CANONICAL_CHECKPOINT` và chấp nhận nhãn NON-CANONICAL.

In [ ]:
KEYPOINT_ROOT = INPUT_DIR / 'keypoint_tensor_cache'
CACHE_ROOT = KEYPOINT_ROOT
INSTANCE_GRAPH_ROOT = INPUT_DIR / 'instance_graphs'
TEMPLATE_DIR = INPUT_DIR / 'template_graphs'
TEST_FILE = INPUT_DIR / TEST_FILE_NAME
if not TEST_FILE.is_file() and TEST_FILE_NAME == 'test_1229.txt':
    fallback_test_file = INPUT_DIR / 'test.txt'
    if fallback_test_file.is_file():
        TEST_FILE = fallback_test_file

TEST_SCRIPT = REPO_DIR / 'final' / 'code' / 'src_neurosymbolic' / 'test_cross_attention_v2.py'

# Chặn nhầm checkpoint được tối ưu theo test.
if CHECKPOINT_PATH.name != 'best.pt' or CHECKPOINT_PATH.name == 'best_test.pt':
    raise ValueError(
        f'Checkpoint phải có đúng tên best.pt (validation-selected), không dùng best_test.pt: {CHECKPOINT_PATH}'
    )
print('CẢNH BÁO đã xác nhận: báo cáo dùng best.pt, không dùng best_test.pt.')

required_directories = {
    'Input': INPUT_DIR,
    'keypoint cache': KEYPOINT_ROOT,
    'instance graphs': INSTANCE_GRAPH_ROOT,
    'template graphs': TEMPLATE_DIR,
    'source code': REPO_DIR / 'final' / 'code' / 'src_neurosymbolic',
}
required_files = {
    'checkpoint best.pt': CHECKPOINT_PATH,
    'test split': TEST_FILE,
    'evaluator script': TEST_SCRIPT,
    'requirements': REQUIREMENTS_PATH,
}
missing_directories = [f'{label}: {path}' for label, path in required_directories.items() if not path.is_dir()]
missing_files = [f'{label}: {path}' for label, path in required_files.items() if not path.is_file()]
if missing_directories or missing_files:
    details = '\n'.join(missing_directories + missing_files)
    raise FileNotFoundError('Thiếu đầu vào bắt buộc:\n' + details)

def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b''):
            digest.update(chunk)
    return digest.hexdigest()

CHECKPOINT_SHA256 = sha256_file(CHECKPOINT_PATH)
TEST_SPLIT_SHA256 = sha256_file(TEST_FILE)
PROVENANCE_STATUS = 'NON-CANONICAL' if ALLOW_NON_CANONICAL_CHECKPOINT else 'CANONICAL'
if CHECKPOINT_SHA256 != EXPECTED_VALIDATION_CHECKPOINT_SHA256 and not ALLOW_NON_CANONICAL_CHECKPOINT:
    raise ValueError(
        'Checkpoint SHA-256 không khớp validation-selected canonical hash. '
        f'expected={EXPECTED_VALIDATION_CHECKPOINT_SHA256}, actual={CHECKPOINT_SHA256}. '
        'Nếu cố ý dùng checkpoint khác, phải bật ALLOW_NON_CANONICAL_CHECKPOINT=True và ghi rõ NON-CANONICAL.'
    )
if ALLOW_NON_CANONICAL_CHECKPOINT:
    print('CẢNH BÁO NON-CANONICAL: cho phép checkpoint ngoài hash canonical.')
else:
    print('Đã xác nhận checkpoint canonical theo SHA-256.')
print('Checkpoint SHA-256:', CHECKPOINT_SHA256)
print('Test split SHA-256:', TEST_SPLIT_SHA256)

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print('Đã kiểm tra đủ đường dẫn bắt buộc.')
print('Test file:', TEST_FILE)
print('Evaluator:', TEST_SCRIPT)
print('Checkpoint validation:', CHECKPOINT_PATH)
print('Provenance status:', PROVENANCE_STATUS)

## 4. Chạy evaluator đúng một lần cho cả ba nhánh

Script sẽ xuất `test_metrics.json`, `test_predictions.csv`, `test_per_class.csv` và `test_confusion_matrices.json` cạnh file metrics. Một lần chạy đã tính đồng thời `final`, `neural` và `kg`; không chạy ba lần với ba checkpoint/nhánh khác nhau.

In [ ]:
import torch

METRICS_PATH = RESULTS_DIR / 'test_metrics.json'
DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'
if DEVICE == 'cpu':
    print('Cảnh báo: CUDA không khả dụng; evaluator sẽ chạy CPU và có thể lâu hơn.')

evaluation_command = [
    sys.executable, '-u', str(TEST_SCRIPT),
    '--checkpoint', str(CHECKPOINT_PATH),
    '--keypoint-root', str(KEYPOINT_ROOT),
    '--cache-root', str(CACHE_ROOT),
    '--instance-graph-root', str(INSTANCE_GRAPH_ROOT),
    '--template-dir', str(TEMPLATE_DIR),
    '--test-file', str(TEST_FILE),
    '--split', 'test',
    '--batch-size', str(BATCH_SIZE),
    '--num-workers', str(NUM_WORKERS),
    '--device', DEVICE,
    '--timing-warmup-batches', '2',
    '--timing-repeats', '1',
    '--eval-branch', 'final',
    '--output', str(METRICS_PATH),
]
print('Đang chạy một lần với argv an toàn:')
print(' '.join(repr(part) for part in evaluation_command))
subprocess.run(evaluation_command, cwd=str(REPO_DIR), check=True)
print('Evaluator đã hoàn tất; metrics:', METRICS_PATH)

## 5. Đọc `test_metrics.json` và tóm tắt ba nhánh

Accuracy và macro-F1 của lần chạy hiện tại được giữ ở thang phần trăm để so sánh trực quan với số liệu bài báo. Giá trị gốc trong JSON vẫn được giữ trong bảng CSV kết quả.

In [ ]:
import pandas as pd
from IPython.display import display

if not METRICS_PATH.is_file():
    raise FileNotFoundError(f'Không tìm thấy test_metrics.json: {METRICS_PATH}')
metrics = json.loads(METRICS_PATH.read_text(encoding='utf-8'))
branch_order = ['neural', 'kg', 'final']
missing_branches = [branch for branch in branch_order if branch not in metrics.get('branches', {})]
if missing_branches:
    raise KeyError(f'test_metrics.json thiếu nhánh: {missing_branches}')

branch_rows = []
for branch in branch_order:
    summary = metrics['branches'][branch]
    branch_rows.append({
        'Nhánh': branch,
        'Accuracy (%)': 100.0 * float(summary['accuracy']),
        'Macro-F1 (%)': 100.0 * float(summary['macro_f1']),
        'Đúng / tổng': f"{summary.get('correct', '?')} / {summary.get('total', '?')}",
    })
current_df = pd.DataFrame(branch_rows)
CURRENT_BRANCHES_CSV = RESULTS_DIR / 'current_branches.csv'
current_df.to_csv(CURRENT_BRANCHES_CSV, index=False, encoding='utf-8-sig')
display(current_df.style.format({'Accuracy (%)': '{:.2f}', 'Macro-F1 (%)': '{:.2f}'}))
print('Checkpoint trong metrics:', metrics.get('checkpoint'))
print('Nhánh được evaluator báo cáo:', list(metrics['branches']))
print('CSV tóm tắt:', CURRENT_BRANCHES_CSV)

## 6. Bảng baseline trong bài báo và bảng so sánh

Các mốc tham chiếu phải ghi đúng như yêu cầu của báo cáo: Sparse-ST-GCN `92.94 ± 0.40`, SignBart `88.06 ± 0.76`, ProtoGCN `93.28 ± 0.27`, Proposed paper `96.40 ± 0.17`, và published DATE Transformer `94.22`. Dấu `±` của bài báo là độ dao động được công bố; nhánh hiện tại chỉ có một lần chạy nên không tự gán độ lệch chuẩn.
Nếu `LEGACY_NEURAL_METRICS_PATH` trỏ tới một file metrics tồn tại, bảng sẽ thêm dòng **Neural-only độc lập (legacy, optional)**. Đây chỉ là benchmark nội bộ được đánh giá độc lập, không thay thế ba nhánh trong checkpoint validation-selected; nếu file không tồn tại thì dòng này bị bỏ qua.

In [ ]:
legacy_rows = []
legacy_metrics = None
LEGACY_METRICS_COPY = None
if LEGACY_NEURAL_METRICS_PATH.is_file():
    legacy_metrics = json.loads(LEGACY_NEURAL_METRICS_PATH.read_text(encoding='utf-8'))
    legacy_summary = legacy_metrics
    if 'branches' in legacy_metrics:
        legacy_summary = legacy_metrics['branches'].get('neural', legacy_metrics['branches'].get('final', {}))
    if 'accuracy' not in legacy_summary or 'macro_f1' not in legacy_summary:
        raise KeyError(f'Metrics legacy thiếu accuracy/macro_f1: {LEGACY_NEURAL_METRICS_PATH}')
    legacy_rows.append({
        'Model': 'Neural-only độc lập (legacy, optional)',
        'Accuracy (%)': 100.0 * float(legacy_summary['accuracy']),
        'Accuracy ± (%)': None,
        'Macro-F1 (%)': 100.0 * float(legacy_summary['macro_f1']),
        'Nhóm': 'Internal benchmark (optional)',
    })
    LEGACY_METRICS_COPY = RESULTS_DIR / 'legacy_neural_only_test_metrics.json'
    if LEGACY_NEURAL_METRICS_PATH.resolve() != LEGACY_METRICS_COPY.resolve():
        import shutil
        shutil.copy2(LEGACY_NEURAL_METRICS_PATH, LEGACY_METRICS_COPY)
    print('Đã thêm benchmark neural-only legacy từ:', LEGACY_NEURAL_METRICS_PATH)
else:
    print('Bỏ qua neural-only legacy: không có metrics path tồn tại tại', LEGACY_NEURAL_METRICS_PATH)

paper_baselines = pd.DataFrame([
    {'Model': 'Sparse-ST-GCN', 'Accuracy (%)': 92.94, 'Accuracy ± (%)': 0.40, 'Macro-F1 (%)': None, 'Nhóm': 'Paper'},
    {'Model': 'SignBart', 'Accuracy (%)': 88.06, 'Accuracy ± (%)': 0.76, 'Macro-F1 (%)': None, 'Nhóm': 'Paper'},
    {'Model': 'ProtoGCN', 'Accuracy (%)': 93.28, 'Accuracy ± (%)': 0.27, 'Macro-F1 (%)': None, 'Nhóm': 'Paper'},
    {'Model': 'Proposed paper', 'Accuracy (%)': 96.40, 'Accuracy ± (%)': 0.17, 'Macro-F1 (%)': None, 'Nhóm': 'Paper'},
    {'Model': 'published DATE Transformer', 'Accuracy (%)': 94.22, 'Accuracy ± (%)': None, 'Macro-F1 (%)': None, 'Nhóm': 'Paper'},
])
current_comparison = pd.DataFrame([
    {
        'Model': f"DATE {row['Nhánh']} (current)",
        'Accuracy (%)': row['Accuracy (%)'],
        'Accuracy ± (%)': None,
        'Macro-F1 (%)': row['Macro-F1 (%)'],
        'Nhóm': 'Current run',
    }
    for row in branch_rows
])
comparison_parts = [paper_baselines]
if legacy_rows:
    comparison_parts.append(pd.DataFrame(legacy_rows))
comparison_parts.append(current_comparison)
comparison_df = pd.concat(comparison_parts, ignore_index=True)
COMPARISON_CSV = RESULTS_DIR / 'baseline_comparison.csv'
comparison_df.to_csv(COMPARISON_CSV, index=False, encoding='utf-8-sig')
display(comparison_df.style.format({'Accuracy (%)': '{:.2f}', 'Accuracy ± (%)': '{:.2f}', 'Macro-F1 (%)': '{:.2f}'}))
print('Bảng so sánh:', COMPARISON_CSV)

## 7. Biểu đồ accuracy

Biểu đồ đặt ba nhánh DATE hiện tại cạnh các baseline công bố. Thanh lỗi chỉ áp dụng cho các baseline có `±`; hãy đọc chênh lệch cùng lưu ý về protocol/tập test ở phần mở đầu.

In [ ]:
import matplotlib.pyplot as plt

plot_df = comparison_df.copy()
plot_df['Accuracy error'] = plot_df['Accuracy ± (%)'].fillna(0.0)
bar_colors = [
    '#9ecae1' if group == 'Paper' else
    '#fdae6b' if group == 'Internal benchmark (optional)' else
    '#74c476' if model.endswith('neural (current)') else
    '#31a354' if model.endswith('kg (current)') else
    '#006d2c'
    for model, group in zip(plot_df['Model'], plot_df['Nhóm'])
]
fig, ax = plt.subplots(figsize=(13, 6))
positions = list(range(len(plot_df)))
ax.bar(
    positions,
    plot_df['Accuracy (%)'],
    yerr=plot_df['Accuracy error'],
    capsize=4,
    color=bar_colors,
    edgecolor='#263238',
)
ax.set_xticks(positions)
ax.set_xticklabels(plot_df['Model'], rotation=25, ha='right')
ax.set_ylabel('Accuracy (%)')
ax.set_ylim(0, 100)
ax.set_title('DATE 27 lớp: accuracy hiện tại và baseline công bố')
ax.grid(axis='y', alpha=0.25)
for position, value in zip(positions, plot_df['Accuracy (%)']):
    ax.text(position, float(value) + 1.0, f'{float(value):.2f}', ha='center', va='bottom', fontsize=8)
fig.tight_layout()
ACCURACY_PLOT_PATH = RESULTS_DIR / 'accuracy_comparison.png'
fig.savefig(ACCURACY_PLOT_PATH, dpi=180, bbox_inches='tight')
plt.show()
plt.close(fig)
print('Đã lưu biểu đồ:', ACCURACY_PLOT_PATH)

## 8. Hiển thị CSV per-class và confusion matrix

`test_per_class.csv` cho biết support, recall và F1 theo lớp của cả ba nhánh. `test_confusion_matrices.json` được mở thành bảng và heatmap riêng cho `neural`, `kg`, `final`; không chỉ xem accuracy tổng.

In [ ]:
import numpy as np

PREDICTIONS_CSV = RESULTS_DIR / 'test_predictions.csv'
PER_CLASS_CSV = RESULTS_DIR / 'test_per_class.csv'
CONFUSION_JSON = RESULTS_DIR / 'test_confusion_matrices.json'
artifact_paths = [PREDICTIONS_CSV, PER_CLASS_CSV, CONFUSION_JSON]
missing_artifacts = [path for path in artifact_paths if not path.is_file()]
if missing_artifacts:
    raise FileNotFoundError(f'Thiếu artifact evaluator: {missing_artifacts}')

per_class_df = pd.read_csv(PER_CLASS_CSV)
if len(per_class_df) != EXPECTED_NUM_CLASSES:
    raise ValueError(f'Per-class CSV có {len(per_class_df)} lớp, kỳ vọng {EXPECTED_NUM_CLASSES}.')
print('Nội dung test_per_class.csv:')
display(per_class_df)

predictions_df = pd.read_csv(PREDICTIONS_CSV)
print('Một vài dòng test_predictions.csv:')
display(predictions_df.head(10))

confusion_payload = json.loads(CONFUSION_JSON.read_text(encoding='utf-8'))
class_names = per_class_df.sort_values('class_id')['class_name'].astype(str).tolist()
CONFUSION_PLOT_PATHS = []
for branch in ['neural', 'kg', 'final']:
    matrix = np.asarray(confusion_payload[branch], dtype=int)
    if matrix.shape != (EXPECTED_NUM_CLASSES, EXPECTED_NUM_CLASSES):
        raise ValueError(f'Confusion matrix {branch} có shape {matrix.shape}, không phải 27x27.')
    print(f'Confusion matrix — {branch}:')
    display(pd.DataFrame(matrix, index=class_names, columns=class_names))
    fig, ax = plt.subplots(figsize=(12, 10))
    image = ax.imshow(matrix, interpolation='nearest', cmap='Blues')
    fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
    ticks = np.arange(EXPECTED_NUM_CLASSES)
    ax.set_xticks(ticks)
    ax.set_yticks(ticks)
    ax.set_xticklabels(class_names, rotation=90, fontsize=6)
    ax.set_yticklabels(class_names, fontsize=6)
    ax.set_xlabel('Dự đoán')
    ax.set_ylabel('Nhãn thật')
    ax.set_title(f'Confusion matrix — nhánh {branch}')
    fig.tight_layout()
    plot_path = RESULTS_DIR / f'confusion_{branch}.png'
    fig.savefig(plot_path, dpi=180, bbox_inches='tight')
    CONFUSION_PLOT_PATHS.append(plot_path)
    plt.show()
    plt.close(fig)
print('Đã lưu heatmap:', CONFUSION_PLOT_PATHS)

## 9. Đóng gói kết quả

Zip cuối cùng chứa metrics, prediction CSV, per-class CSV, confusion matrix JSON, bảng so sánh, manifest và các hình PNG để gửi kèm báo cáo. Nếu có metrics legacy, bản sao của file đó cũng được đóng gói. Manifest ghi URL/commit, checkpoint, đường dẫn dữ liệu và thời điểm chạy.

In [ ]:
import zipfile

MANIFEST_PATH = RESULTS_DIR / 'run_manifest.json'
manifest = {
    'created_at_utc': dt.datetime.now(dt.timezone.utc).isoformat(),
    'repo_url': REPO_URL,
    'repo_commit': REPO_COMMIT_RESOLVED,
    'checkpoint': str(CHECKPOINT_PATH),
    'checkpoint_sha256': CHECKPOINT_SHA256,
    'expected_validation_checkpoint_sha256': EXPECTED_VALIDATION_CHECKPOINT_SHA256,
    'test_split_sha256': TEST_SPLIT_SHA256,
    'provenance_status': PROVENANCE_STATUS,
    'checkpoint_policy': 'best.pt selected by validation; best_test.pt is prohibited',
    'input_dir': str(INPUT_DIR),
    'test_file': str(TEST_FILE),
    'metrics': str(METRICS_PATH),
    'legacy_neural_metrics_path': str(LEGACY_NEURAL_METRICS_PATH),
    'legacy_neural_metrics_included': bool(legacy_rows),
    'branches': branch_order,
    'expected_num_classes': EXPECTED_NUM_CLASSES,
}
MANIFEST_PATH.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')

REPORT_ZIP = RESULTS_DIR.with_suffix('.zip')
files_to_archive = sorted(path for path in RESULTS_DIR.rglob('*') if path.is_file())
with zipfile.ZipFile(REPORT_ZIP, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for path in files_to_archive:
        archive.write(path, arcname=path.relative_to(RESULTS_DIR))

print('Đã tạo gói kết quả:', REPORT_ZIP)
print('Số file trong zip:', len(files_to_archive))
print('Các file chính:', [path.name for path in files_to_archive])

## 10. Ghi chú khi đưa vào báo cáo

* Dùng dòng `DATE final (current)` làm kết quả fusion chính; giữ `DATE neural (current)` và `DATE kg (current)` để giải thích đóng góp của từng nhánh.
* Báo cáo accuracy cùng macro-F1 từ `current_branches.csv`, không suy ra macro-F1 từ accuracy.
* Khi so với Sparse-ST-GCN, SignBart, ProtoGCN, Proposed paper và published DATE Transformer, ghi rõ khác biệt về số lớp, split, tiền xử lý và giao thức đánh giá nếu có.
* Gửi file `REPORT_ZIP` cùng bảng/biểu đồ; không dùng lại `best_test.pt` cho một báo cáo được tuyên bố là validation-selected.